# Uniform-innovation AR(p) zonotopes and intrinsic volumes

This notebook focuses exclusively on stable AR(1) and AR(2) processes with symmetric uniform innovations

$$X_t=c+\phi_1X_{t-1}+\cdots+\phi_pX_{t-p}+\varepsilon_t,\qquad \varepsilon_t\sim\operatorname{Unif}[-a,a].$$

For the forward embedding

$$Y_t^{(k)}=(X_t,X_{t+1},\ldots,X_{t+k-1}),$$

a finite innovation history of length $m$ produces a zonotope

$$Z_{k,m}=\mu\mathbf 1_k+G_{k,m}[-1/2,1/2]^{m+k}.$$

The columns of $G_{k,m}$ are scaled impulse-response vectors. For a stable AR process, $Z_{k,m}$ approaches the stationary support zonoid as $m\to\infty$.

For a zonotope generated by columns $g_1,\ldots,g_N\in\mathbb R^k$, its intrinsic volumes satisfy

$$V_j(Z)=\sum_{|I|=j}\sqrt{\det(G_I^\top G_I)},\qquad j=1,\ldots,k,\qquad V_0(Z)=1.$$

Thus $V_1$ is the sum of generator lengths, while $V_k$ is the ordinary $k$-dimensional volume. The determinant sum is evaluated exactly when feasible and estimated by uniform subset sampling otherwise. The top-dimensional zonotope-volume identity is associated with McMullen's work on [volumes of projections of cubes](https://doi.org/10.1112/blms/16.3.278).

In [1]:
# If needed, uncomment once:
# %pip install numpy scipy matplotlib plotly ipywidgets

import ast
import math
from itertools import combinations, product

import numpy as np
import matplotlib.pyplot as plt
import plotly.graph_objects as go
import ipywidgets as widgets

from IPython.display import display, clear_output
from scipy.spatial import ConvexHull

plt.style.use("seaborn-v0_8-whitegrid")

## AR impulse responses and zonotope generators

In [2]:
def companion_matrix(phi):
    phi = np.asarray(phi, dtype=float)
    p = len(phi)
    matrix = np.zeros((p, p))
    matrix[0] = phi
    if p > 1:
        matrix[1:, :-1] = np.eye(p - 1)
    return matrix


def check_stationarity(phi):
    eigenvalues = np.linalg.eigvals(companion_matrix(phi))
    if np.any(np.abs(eigenvalues) >= 1):
        raise ValueError(
            "The AR coefficients are not stationary. Companion eigenvalue magnitudes: "
            f"{np.round(np.abs(eigenvalues), 5)}"
        )
    return eigenvalues


def parse_coefficients(text, order):
    text = text.strip()
    values = ast.literal_eval(text) if text.startswith(("[", "(")) else [
        float(item.strip()) for item in text.split(",")
    ]
    phi = np.atleast_1d(np.asarray(values, dtype=float))
    if len(phi) != order:
        raise ValueError(f"AR({order}) requires exactly {order} coefficients.")
    check_stationarity(phi)
    return phi


def ar_impulse_response(phi, maximum_lag):
    """Return psi_0,...,psi_maximum_lag for 1/phi(B)."""
    phi = np.asarray(phi, dtype=float)
    psi = np.zeros(maximum_lag + 1)
    psi[0] = 1.0
    for lag in range(1, maximum_lag + 1):
        psi[lag] = sum(
            phi[j - 1] * psi[lag - j]
            for j in range(1, min(len(phi), lag) + 1)
        )
    return psi


def ar_window_zonotope_generators(phi, embedding_dimension, history, half_width=1.0):
    """Generator matrix for the finite-history stationary-window approximation.

    Innovations have relative times s=-history,...,embedding_dimension-1.
    A generator is twice the innovation half-width times its impulse response
    in each coordinate, because the coefficient cube is [-1/2,1/2]^N.
    """
    if history < 0 or embedding_dimension < 1 or half_width <= 0:
        raise ValueError("Require history >= 0, embedding_dimension >= 1, half_width > 0.")
    phi = np.asarray(phi, dtype=float)
    check_stationarity(phi)
    psi = ar_impulse_response(phi, history + embedding_dimension - 1)
    innovation_times = np.arange(-history, embedding_dimension)
    generators = np.zeros((embedding_dimension, len(innovation_times)))
    for column, innovation_time in enumerate(innovation_times):
        for coordinate in range(embedding_dimension):
            lag = coordinate - innovation_time
            if lag >= 0:
                generators[coordinate, column] = 2.0 * half_width * psi[lag]
    nonzero = np.linalg.norm(generators, axis=0) > 1e-14
    return generators[:, nonzero], innovation_times[nonzero], psi


def simulate_uniform_ar(phi, n=3000, half_width=1.0, intercept=0.0,
                        burnin=1000, seed=7):
    phi = np.asarray(phi, dtype=float)
    check_stationarity(phi)
    rng = np.random.default_rng(seed)
    p = len(phi)
    total = n + burnin + p
    innovations = rng.uniform(-half_width, half_width, total)
    stationary_mean = intercept / (1.0 - np.sum(phi))
    x = np.full(total, stationary_mean)
    for t in range(p, total):
        x[t] = intercept + phi @ x[t - np.arange(1, p + 1)] + innovations[t]
    return x[burnin + p:]


def forward_embedding(x, dimension):
    x = np.asarray(x)
    return np.column_stack([
        x[offset:len(x) - dimension + 1 + offset]
        for offset in range(dimension)
    ])

## Exact and approximate intrinsic volumes

When $\binom Nj$ exceeds the selected exact-computation limit, subsets are sampled uniformly. The reported estimate is unbiased for the determinant sum, and `standard_error` is its Monte Carlo standard error. The same random seed makes results reproducible.

In [3]:
def parallelotope_j_volume(generator_subset):
    """j-volume spanned by the columns of a k-by-j matrix."""
    gram = generator_subset.T @ generator_subset
    sign, log_determinant = np.linalg.slogdet(gram)
    if sign <= 0:
        return 0.0
    return float(np.exp(0.5 * log_determinant))


def intrinsic_volume_of_zonotope(generators, order, exact_limit=200_000,
                                  monte_carlo_samples=30_000, seed=7):
    """Compute V_order exactly or estimate it by random generator subsets."""
    generators = np.asarray(generators, dtype=float)
    ambient_dimension, number_of_generators = generators.shape
    if order == 0:
        return {
            "value": 1.0, "standard_error": 0.0, "exact": True,
            "total_subsets": 1, "evaluated_subsets": 1,
        }
    if not 1 <= order <= ambient_dimension:
        raise ValueError("order must lie between 0 and the ambient dimension.")
    total_subsets = math.comb(number_of_generators, order)

    if order == 1:
        value = float(np.linalg.norm(generators, axis=0).sum())
        return {
            "value": value, "standard_error": 0.0, "exact": True,
            "total_subsets": total_subsets, "evaluated_subsets": total_subsets,
        }

    if total_subsets <= exact_limit:
        values = np.fromiter(
            (parallelotope_j_volume(generators[:, subset])
             for subset in combinations(range(number_of_generators), order)),
            dtype=float,
            count=total_subsets,
        )
        return {
            "value": float(values.sum()), "standard_error": 0.0, "exact": True,
            "total_subsets": total_subsets, "evaluated_subsets": total_subsets,
        }

    sample_count = min(int(monte_carlo_samples), total_subsets)
    rng = np.random.default_rng(seed + 1009 * order + 9176 * ambient_dimension)
    values = np.empty(sample_count)
    for sample in range(sample_count):
        subset = rng.choice(number_of_generators, size=order, replace=False)
        values[sample] = parallelotope_j_volume(generators[:, subset])
    estimate = total_subsets * values.mean()
    standard_error = total_subsets * values.std(ddof=1) / np.sqrt(sample_count)
    return {
        "value": float(estimate), "standard_error": float(standard_error),
        "exact": False, "total_subsets": total_subsets,
        "evaluated_subsets": sample_count,
    }


def zonotope_intrinsic_volume_sequence(phi, maximum_dimension=8, history=15,
                                        half_width=1.0, exact_limit=200_000,
                                        monte_carlo_samples=30_000, seed=7):
    results = {}
    values = np.full((maximum_dimension + 1, maximum_dimension + 1), np.nan)
    errors = np.full_like(values, np.nan)
    exact = np.zeros_like(values, dtype=bool)

    for dimension in range(1, maximum_dimension + 1):
        generators, innovation_times, psi = ar_window_zonotope_generators(
            phi, dimension, history, half_width
        )
        dimension_results = []
        for order in range(dimension + 1):
            result = intrinsic_volume_of_zonotope(
                generators, order, exact_limit=exact_limit,
                monte_carlo_samples=monte_carlo_samples, seed=seed
            )
            dimension_results.append(result)
            values[dimension, order] = result["value"]
            errors[dimension, order] = result["standard_error"]
            exact[dimension, order] = result["exact"]
        results[dimension] = {
            "generators": generators,
            "innovation_times": innovation_times,
            "impulse_response": psi,
            "intrinsic_volumes": dimension_results,
        }
    return results, values, errors, exact


def cube_validation(dimension=5, half_width=1.3):
    """For G=2a I, V_j must equal binom(d,j)(2a)^j."""
    generators = 2.0 * half_width * np.eye(dimension)
    numerical = np.array([
        intrinsic_volume_of_zonotope(generators, order)["value"]
        for order in range(dimension + 1)
    ])
    exact = np.array([
        math.comb(dimension, order) * (2.0 * half_width) ** order
        for order in range(dimension + 1)
    ])
    return float(np.max(np.abs(numerical - exact) / np.maximum(exact, 1e-15)))

## Visualization and diagnostics

## Interactive experiment

The defaults keep most determinant sums exact. Increasing `maximum k` or history can make $\binom{m+k}{j}$ enormous; the subset cap then activates Monte Carlo estimation. The 3D zonotope surface is only a visualization based on random zonotope samples—the intrinsic volumes are computed directly from generators, not from this mesh.

In [5]:
order_widget = widgets.ToggleButtons(options=[1, 2], value=1, description="AR order:")
phi_widget = widgets.Text(value="0.7", description="Coefficients:",
                          style={"description_width": "120px"})
half_width_widget = widgets.FloatSlider(value=1.0, min=0.1, max=3.0, step=0.1,
                                        description="Uniform half-width:",
                                        style={"description_width": "140px"})
history_widget = widgets.IntSlider(value=12, min=0, max=50, step=1,
                                   description="History m:", style={"description_width": "120px"})
max_dimension_widget = widgets.IntSlider(value=7, min=2, max=12, step=1,
                                         description="Maximum k:", style={"description_width": "120px"})
sample_size_widget = widgets.BoundedIntText(value=5000, min=500, max=500000, step=1000,
                                            description="Simulation n:",
                                            style={"description_width": "120px"})
exact_limit_widget = widgets.Dropdown(options=[50_000, 200_000, 1_000_000], value=200_000,
                                     description="Exact subset cap:",
                                     style={"description_width": "130px"})
mc_samples_widget = widgets.Dropdown(options=[5_000, 15_000, 30_000, 75_000], value=30_000,
                                    description="MC subsets:",
                                    style={"description_width": "130px"})
seed_widget = widgets.IntText(value=7, description="Seed:",
                              style={"description_width": "120px"})
run_button = widgets.Button(description="Generate and compute", button_style="primary", icon="play")
output = widgets.Output()


def update_coefficient_default(change):
    phi_widget.value = "0.7" if change["new"] == 1 else "0.6, -0.2"


order_widget.observe(update_coefficient_default, names="value")


def run_uniform_zonotope_experiment(_=None):
    with output:
        clear_output(wait=True)
        try:
            phi = parse_coefficients(phi_widget.value, order_widget.value)
            relative_error = cube_validation()
            print(f"Cube-formula validation: maximum relative error = {relative_error:.2e}")
            print(f"Companion eigenvalues: {np.round(np.linalg.eigvals(companion_matrix(phi)), 5)}")

            global uniform_ar_series, zonotope_results, intrinsic_values
            global intrinsic_errors, intrinsic_exact
            uniform_ar_series = simulate_uniform_ar(
                phi, n=sample_size_widget.value,
                half_width=half_width_widget.value, seed=seed_widget.value
            )
            plot_embeddings_and_zonotopes(
                uniform_ar_series, phi, history_widget.value,
                half_width_widget.value, seed_widget.value
            )
            zonotope_results, intrinsic_values, intrinsic_errors, intrinsic_exact = (
                zonotope_intrinsic_volume_sequence(
                    phi, maximum_dimension=max_dimension_widget.value,
                    history=history_widget.value,
                    half_width=half_width_widget.value,
                    exact_limit=exact_limit_widget.value,
                    monte_carlo_samples=mc_samples_widget.value,
                    seed=seed_widget.value,
                )
            )
            plot_intrinsic_volume_sequence(
                intrinsic_values, intrinsic_errors, intrinsic_exact,
                max_dimension_widget.value
            )

            histories = np.unique(np.linspace(0, history_widget.value, 7, dtype=int))
            plot_history_convergence(
                phi, min(3, max_dimension_widget.value), histories,
                half_width_widget.value, exact_limit_widget.value,
                mc_samples_widget.value, seed_widget.value
            )

            print("\nIntrinsic-volume summary")
            print(" k     V1             V2             Vk          method for Vk")
            for k in range(1, max_dimension_widget.value + 1):
                v1 = intrinsic_values[k, 1]
                v2 = intrinsic_values[k, min(2, k)]
                vk = intrinsic_values[k, k]
                method = "exact" if intrinsic_exact[k, k] else "Monte Carlo"
                print(f"{k:2d}  {v1:12.5g}  {v2:12.5g}  {vk:12.5g}  {method}")
        except Exception as exc:
            print(f"Could not complete the experiment: {exc}")


run_button.on_click(run_uniform_zonotope_experiment)
display(widgets.VBox([
    widgets.HTML("<b>Uniform AR zonotope settings</b>"),
    widgets.HBox([order_widget, phi_widget]),
    widgets.HBox([half_width_widget, history_widget]),
    widgets.HBox([max_dimension_widget, sample_size_widget]),
    widgets.HBox([exact_limit_widget, mc_samples_widget]),
    widgets.HBox([seed_widget, run_button]),
]), output)
run_uniform_zonotope_experiment()

Output()

## Interpretation

- Increasing $k$ changes both the ambient dimension and the number of innovation generators.
- Increasing $m$ adds older impulse-response generators. Stability makes their norms decay, so the intrinsic volumes should stabilize.
- $V_0=1$ for every nonempty convex zonotope.
- $V_1$ is especially cheap and exact: it is simply the sum of all generator lengths.
- $V_k$ is the ordinary volume of the $k$-dimensional support approximation.
- Monte Carlo errors concern only the combinatorial determinant sum; they do not include finite-history truncation error. Use the history-convergence plot to assess the latter.